
# C6-pytorch — Session 1: Torch Tensors and Python Inheritance

*One class session, roughly 80 minutes. Prerequisites: C5-neural-networks
(the perceptron, threshold and ReLU activations, MLP forward passes with
hand-set weights) and, through it, the NumPy craft of F1: array creation,
shapes, dtypes, broadcasting, axis aggregation, seeded randomness.*

**This session:** C5 built every network out of NumPy arrays.
This unit moves the same networks into **PyTorch**, the library the real
exam hands you, and the move happens in two steps that this session
teaches side by side.
First, torch's array object — the **tensor** — which mirrors the NumPy
array so closely that most of your F1 reflexes transfer after a rename.
Second, a piece of the Python language itself that PyTorch is built on:
**classes and inheritance**, taught here with plain-Python examples
before any torch appears, because `nn.Module` — next session's star — is
nothing but a base class you inherit from.
Everything runs on the CPU; nothing in this unit needs a GPU.


In [ ]:

import numpy as np
import torch

print("torch", torch.__version__)



## 1. Tensors: NumPy Arrays with a Passport

**Motivation.**
Why learn a second array library when F1's NumPy already does everything
this course computes?
Because the neural-network ecosystem — including the exam's starter code —
is written in PyTorch: its networks, its saved weights, and its model
zoo (C7 will download one) all speak **tensor**.
The good news is that a `torch.Tensor` behaves so much like an
`np.ndarray` that this section is mostly a translation table.

Creation mirrors F1 line for line:

| F1 NumPy | torch | note |
|---|---|---|
| `np.array([1.0, 2.0])` | `torch.tensor([1.0, 2.0])` | list → array/tensor |
| `np.zeros((3, 2))` | `torch.zeros(3, 2)` | torch also accepts a tuple |
| `np.ones((2, 4))` | `torch.ones(2, 4)` | |
| `np.arange(6)` | `torch.arange(6)` | |
| `np.linspace(0, 1, 5)` | `torch.linspace(0, 1, 5)` | |
| `a.shape`, `a.ndim` | `t.shape`, `t.ndim` | identical |
| `a.reshape(2, 3)` | `t.reshape(2, 3)` | identical |
| `a[1, 2]`, `a[:, 0]` | `t[1, 2]`, `t[:, 0]` | indexing/slicing identical |

One genuinely new habit: a single entry of a tensor, like `t[1, 2]`, is
still a (0-dimensional) tensor.
To get the plain Python number out, call `.item()`.


In [ ]:

t = torch.arange(6).reshape(2, 3)
print(t)
print("shape:", t.shape, " ndim:", t.ndim)
print("t[1, 2] as a tensor:", t[1, 2])
print("t[1, 2] as a number:", t[1, 2].item())
print("a column:", t[:, 0])



The printout says `tensor([[0, 1, 2], [3, 4, 5]])` — same layout, same
row-major filling, same slicing as the NumPy `arange(6).reshape(2, 3)`
you have used since F1.
`t.shape` prints as `torch.Size([2, 3])`, which behaves like the tuple
`(2, 3)` in every way you will need (indexing, comparison, unpacking).

### Checkpoint 1

1. Without running it: what are the shape, `ndim`, and the value of
   `u[1, 3]` for `u = torch.arange(10).reshape(2, 5)`?
2. Translate to torch, keeping the same values: `np.zeros((4, 2))` and
   `b.sum(axis=1)` for a 2-D array `b`. *(Peek ahead: torch spells the
   axis keyword differently — guess, then check in Section 3.)*
3. `v = torch.tensor([[1.0, 2.0], [3.0, 4.0]])`: what does `v[0, 1]`
   print, and how do you extract it as a plain Python `float`?



## 2. Dtypes: the float32 Border Crossing

**Motivation.**
The first real difference between the libraries is invisible until it
breaks an assert.
NumPy's default floating dtype is `float64`; **torch's is `float32`**,
half the bytes and (on big models) a large speed and memory win — which
is why the deep-learning world accepts the lower precision.
Integer literals become `int64` in both libraries.


In [ ]:

print("torch float default:", torch.tensor([1.0, 2.0]).dtype)
print("torch int default:  ", torch.tensor([1, 2]).dtype)
print("numpy float default:", np.array([1.0, 2.0]).dtype)



That prints `torch.float32`, `torch.int64`, `float64` — the mismatch in
the first and last lines is the border you must learn to cross on
purpose, never by accident.
Watch what float32 actually stores when you ask it for $0.1$:


In [ ]:

x32 = torch.tensor(0.1)                 # float32
x64 = np.float64(0.1)
print(f"float32 0.1 stores {x32.item():.20f}")
print(f"float64 0.1 stores {x64:.20f}")

# a million float32 additions drift visibly; float64 stays clean
print("float32 sum of 10^6 copies of 0.1:", torch.full((1000000,), 0.1).sum())
print("float64 sum of 10^6 copies of 0.1:", torch.full((1000000,), 0.1, dtype=torch.float64).sum())



The float32 copy of $0.1$ is wrong in the 9th digit
($0.10000000149\ldots$ vs float64's $0.10000000000\ldots$), and a
million additions pile that up into `100000.0078` — off by nearly $0.01$
where the float64 sum shows `100000.0000`.

**Two course conventions follow, both used in every notebook of this
unit:**

1. **Work in float64 by default.**
   `torch.set_default_dtype(torch.float64)` makes every new *float*
   tensor float64, so torch results agree with this course's NumPy
   results to the precision our asserts expect.
   (It changes only floating-point defaults — integer tensors stay
   `int64` regardless.)
2. **When float32 is used deliberately** — as in the demo above, or when
   C7's downloaded models force it — comparisons state their tolerance
   explicitly: `atol=1e-6`, `rtol=1e-5`.
   Never expect float32 to match float64 to 12 digits; it physically
   cannot.

Casting between dtypes is `.to(...)`:


In [ ]:

torch.set_default_dtype(torch.float64)   # course convention from here on

print("new float default:", torch.tensor([1.0, 2.0]).dtype)
print("ints unaffected:  ", torch.tensor([1, 2]).dtype)

k = torch.arange(4)                      # int64
kf = k.to(torch.float64)                 # explicit cast
print(k.dtype, "->", kf.dtype, kf)



### Checkpoint 2

1. Before any `set_default_dtype` call: what are the dtypes of
   `torch.tensor([3, 4])` and `torch.tensor([3.0, 4])`?
2. A check like `abs(value - expected) < 1e-12` passes for a float64
   computation but fails for the same computation in float32.
   Why, and what are the course's two sanctioned fixes?
3. After `torch.set_default_dtype(torch.float64)`, what is the dtype of
   `torch.tensor([1, 2])`, and why?



## 3. Arithmetic, Broadcasting, and `dim=`

**Motivation.**
Everything F1 taught about *computing* with arrays transfers: elementwise
arithmetic, comparisons, broadcasting, and axis aggregation all work the
same way — with one rename.
What NumPy calls `axis`, torch calls **`dim`**.

| F1 NumPy | torch |
|---|---|
| `a + b`, `a * b`, `a ** 2` | identical |
| `a > 0` (boolean array) | `t > 0` (boolean tensor) |
| `mask.astype(float)` | `mask.to(t.dtype)` |
| `a.sum(axis=0)` | `t.sum(dim=0)` |
| `a.mean(axis=1)` | `t.mean(dim=1)` |
| `a.max(axis=1)` | `t.max(dim=1)` — returns `.values` *and* `.indices` |
| `a.argmax(axis=0)` | `t.argmax(dim=0)` |
| `a @ b` (matrix product) | `t @ u` — identical |
| `a.T` | `t.T` — identical |

Two entries deserve a second look.
`t.max(dim=1)` returns a *pair* (the max values and where they sit) —
take `.values` when you only want the maxima.
And boolean masks are cast with `.to(...)`, exactly the move C5's step
activation made with `astype`.


In [ ]:

t = torch.arange(12).reshape(3, 4)
print(t)
print("column sums (dim=0):", t.sum(dim=0))
print("row sums    (dim=1):", t.sum(dim=1))
print("row maxes:          ", t.max(dim=1).values)
print("multiples of 3:     ", int((t % 3 == 0).sum()))



Read the aggregation direction exactly as in F1: `dim=0` collapses the
rows (one result per column — `[12, 15, 18, 21]`), `dim=1` collapses the
columns (one result per row — `[6, 22, 38]`), and the boolean-mask count
finds the 4 multiples of 3 in `0..11` (0, 3, 6, 9).

Broadcasting follows F1's rules unchanged — a `(3,)` row vector
stretches across a `(2, 3)` matrix:


In [ ]:

M = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
shift = torch.tensor([10.0, 20.0, 30.0])
print(M + shift)                       # (2,3) + (3,) -> (2,3)
print((M - M.mean(dim=0)))             # center each column



### Checkpoint 3

1. By hand: `torch.tensor([[1., 2., 3.], [4., 5., 6.]]).sum(dim=1)` —
   values and shape?
2. `A` has shape `(5, 3)` and `v` has shape `(3,)`.
   What is the shape of `A * v`, and which F1 rule says so?
3. For `t = torch.arange(12).reshape(3, 4)`: write a one-liner counting
   the entries strictly greater than 5, and give its value.



## 4. Seeds and the NumPy Bridge

**Randomness.**
Torch draws random tensors with `torch.randn` (standard normals),
`torch.rand` (uniform on $[0,1)$), and `torch.randint`.
Where F1 seeded a *generator object* (`rng = np.random.default_rng(SEED)`),
torch's everyday idiom seeds a *global* stream:
`torch.manual_seed(SEED)`.
This course's pinned seed is `20260804`, same as the NumPy units, and
the discipline is: **re-seed immediately before every random draw you
want reproducible.**
The seed sets the stream's starting point — consecutive draws continue
the stream, so only a fresh `manual_seed` replays the same numbers.


In [ ]:

torch.manual_seed(20260804)
a = torch.randn(2, 3)
b = torch.randn(2, 3)                  # continues the stream: different numbers
torch.manual_seed(20260804)
c = torch.randn(2, 3)                  # re-seeded: replays a exactly
print(a)
print("b equals a:", bool((b == a).all()))
print("c equals a:", bool((c == a).all()))



The first draw prints
`[[ 0.2394, -1.1592, -0.4109], [-0.2183, -0.1205, 0.7447]]`;
`b` differs from `a` (the stream moved on) while `c` reproduces `a`
exactly (`True`) because the seed was reset.

**Crossing to and from NumPy.**
Two one-liners connect the libraries, and one of them has a sharp edge:

- `torch.from_numpy(a)` wraps a NumPy array as a tensor **sharing the
  same memory** — change one, you change the other;
- `t.numpy()` goes the other way, also sharing memory.


In [ ]:

arr = np.zeros(3)
view = torch.from_numpy(arr)
arr[0] = 7.0                            # write through the NumPy side
print("tensor sees it:", view)

back = view.numpy()
print("round trip is the same memory:", back is not arr, "->", np.shares_memory(back, arr))



Writing `7.0` into the array makes the tensor print `[7., 0., 0.]`
without any copy — shared memory, confirmed by
`np.shares_memory(...) = True`.
When you want an independent copy, say so: `t.clone()` on the torch
side, `np.array(t)` on the NumPy side.

**The border rule.**
`from_numpy` inherits NumPy's dtype (float64), while torch code you
write before `set_default_dtype` may hold float32 — and mixed-dtype
matrix products raise errors.
So this course **casts explicitly at every NumPy/torch boundary**,
e.g. `torch.from_numpy(arr).to(torch.float64)` or
`np.allclose(t.numpy(), arr)` only after making both sides float64.

**Devices, briefly.**
Every tensor also carries a `device` (`t.device` prints `cpu` here).
On machines with a GPU, tensors can move to `cuda` devices for speed;
this course and its exam problems run entirely on the CPU, so `cpu` is
the only device you will see, and no code in this unit mentions devices
again.

### Checkpoint 4

1. `arr = np.ones(2)`; `t = torch.from_numpy(arr)`; then `arr[1] = 5.0`.
   What does `t` print, and why?
2. `torch.manual_seed(20260804)` is run once, followed by two
   `torch.randn(3)` calls. Are the two draws equal?
   What must you do to make the second draw replay the first?
3. Why does the course write
   `torch.from_numpy(arr).to(torch.float64)` with the explicit `.to`,
   even though `from_numpy` already yields float64 here?
   *(Hint: what happens when someone later feeds this code a float32
   array?)*



## 5. Classes in Ten Minutes (Plain Python)

**Motivation.**
Put torch aside for two sections — what follows is pure Python, and it
is the syllabus's real target here.
PyTorch asks you to package every network as a *class* that *inherits*
from `nn.Module`.
To read or write that sentence you need the two underlined words, so we
build them with no tensors in sight.

**A class is a blueprint; an object is one house built from it.**
The blueprint lists the data each object carries (**attributes**) and
the functions that operate on it (**methods**).
`__init__` runs once at construction time and stores the attributes on
`self` — the object being built.
Every method's first parameter is `self`; Python fills it in
automatically when you call `obj.method(...)`.


In [ ]:

class Thermometer:
    """Remembers readings and reports statistics about them."""

    def __init__(self, unit):
        self.unit = unit          # attribute: fixed at construction
        self.readings = []        # attribute: grows over time

    def record(self, value):
        self.readings.append(value)

    def summary(self):
        n = len(self.readings)
        mean = sum(self.readings) / n
        return f"{n} readings, mean {mean:.1f} {self.unit}"


lab = Thermometer("C")            # __init__ runs here
lab.record(20.0)
lab.record(24.0)
print(lab.summary())
print("attributes:", lab.unit, lab.readings)



`lab.summary()` prints `2 readings, mean 22.0 C`.
Notice the call carries no arguments even though `summary` declares
`self`: `lab.summary()` is Python shorthand for
`Thermometer.summary(lab)` — the object slides into the `self` slot.
That one fact demystifies most class syntax.

### Checkpoint 5

1. Write (on paper) a class `Counter` whose `__init__` sets `self.n = 0`
   and whose method `bump()` adds 1 to `self.n` and returns it.
   What do two successive `c.bump()` calls return?
2. In `lab.record(24.0)`: which object becomes `self`, and which
   attribute changes?
3. What is the difference between `lab.unit` and `lab.summary` — and
   what do the parentheses in `lab.summary()` do?



## 6. Inheritance and `super()`

**Motivation.**
Classes become powerful when one class can *extend* another.
**Inheritance** lets a new class (the **subclass**) reuse everything an
existing class (the **base class**) defines, replacing only what must
differ.
The syntax is the parenthesis in `class Sub(Base):`.

The base class below is a template for "things that transform numbers":
it stores a name, promises an `apply` method, and — the important part —
defines `twice` and `label` *once*, in terms of whatever `apply` a
subclass provides.


In [ ]:

class Transform:
    """Base class: stores a name, defines behavior shared by all transforms."""

    def __init__(self, name):
        self.name = name

    def apply(self, x):
        raise NotImplementedError("subclasses must define apply")

    def twice(self, x):
        return self.apply(self.apply(x))   # written once, works for every subclass

    def label(self):
        return f"{self.name}-transform"


class Shift(Transform):
    def __init__(self, c):
        super().__init__("shift")          # let the base class do its setup
        self.c = c

    def apply(self, x):                    # override: this replaces the base apply
        return x + self.c


sh = Shift(3.0)
print(sh.label(), "| apply(1.0) =", sh.apply(1.0), "| twice(1.0) =", sh.twice(1.0))
print("isinstance checks:", isinstance(sh, Shift), isinstance(sh, Transform))



Read the printout against the mechanism:

- `sh.apply(1.0) = 4.0` runs **`Shift.apply`** — the subclass
  **overrides** the base method of the same name;
- `sh.twice(1.0) = 7.0` runs **`Transform.twice`**, which `Shift` never
  defined — the subclass **inherits** it, and inside it each
  `self.apply` call finds the *subclass's* version ($1 \to 4 \to 7$);
- `sh.label()` prints `shift-transform` — inherited too, using the name
  that `super().__init__("shift")` stored;
- `isinstance(sh, Transform)` is `True`: a `Shift` *is a* `Transform`.

**`super().__init__(...)` is the load-bearing line.**
`super()` refers to the base class; calling its `__init__` first gives
the base class its chance to set up the attributes *its* methods rely
on (here `self.name`, which `label` reads).
Skip it and the object is half-built:


In [ ]:

class BrokenShift(Transform):
    def __init__(self, c):
        self.c = c                         # BROKEN: never calls super().__init__

    def apply(self, x):
        return x + self.c


bad = BrokenShift(3.0)
print("apply still works:", bad.apply(1.0))
try:
    bad.label()
except AttributeError as e:
    print("label() explodes -> AttributeError:", e)



`apply` happens to work (it only needs `self.c`), but `label()` raises
`AttributeError: 'BrokenShift' object has no attribute 'name'` — the
base class never got to store it.
The bug is treacherous because it detonates far from its cause; the fix
is mechanical: **a subclass `__init__` calls `super().__init__(...)`
first, before its own setup.**
Next session you will see that `nn.Module` enforces this rule loudly
rather than letting you limp along.

### Checkpoint 6

1. Add `class Scale(Transform)` with factor `k` stored via
   `super().__init__("scale")` and `apply(x) = k * x`.
   For `Scale(2.0)`: what do `apply(3.0)`, `twice(3.0)`, and `label()`
   return?
2. In `sh.twice(1.0)`, the body of `twice` lives in `Transform` —
   which class's `apply` does it call, and what language rule decides?
3. Predict both `isinstance(Scale(2.0), Shift)` and
   `isinstance(Scale(2.0), Transform)`.



## 7. The Payoff: `nn.Module` Is a Base Class

**Motivation.**
Now put the two halves of this session together.
`torch.nn` ships a base class, **`nn.Module`**, that plays exactly the
role `Transform` just played: it defines a large amount of shared
machinery once, and asks each subclass to fill in one method — called
**`forward`** — that says what the module computes.
Here is the smallest possible example, a module that doubles its input:


In [ ]:

import torch.nn as nn


class Doubler(nn.Module):
    def __init__(self):
        super().__init__()                 # the mandatory first line

    def forward(self, x):
        return 2 * x


d = Doubler()
x = torch.tensor([1.0, -2.0, 0.5])
print("d(x)        =", d(x))
print("d.forward(x) =", d.forward(x))
print("isinstance of nn.Module:", isinstance(d, nn.Module))



Map every line onto Section 6:

- `class Doubler(nn.Module)` — subclass syntax, base class `nn.Module`;
- `super().__init__()` — the base class sets up its internal bookkeeping
  (next session shows what it tracks);
- `forward` — the one method you owe, like `Transform`'s `apply`;
- `d(x)` — you *call the object like a function* and your `forward`
  runs.
  That trick is inherited machinery: `nn.Module` defines the special
  method `__call__`, which (after its own housekeeping) hands your
  arguments to `forward`.
  The convention everywhere in torch is to write `d(x)`, never
  `d.forward(x)` — same result here, but only the former passes through
  the base class's machinery, and later units rely on that.

And `nn.Module` polices the `super().__init__()` rule from Section 6:


In [ ]:

class Impatient(nn.Module):
    def __init__(self):
        # BROKEN: tries to attach torch state before nn.Module is initialized
        self.w = nn.Parameter(torch.zeros(2), requires_grad=False)


try:
    Impatient()
except AttributeError as e:
    print("AttributeError:", e)



`AttributeError: cannot assign parameters before Module.__init__() call`
— the same half-built-object failure as `BrokenShift`, except torch
raises it immediately at the assignment instead of letting `label()`
explode later.
(`nn.Parameter` and that `requires_grad=False` are Session 2's
subject — for now, read the demo only as `super().__init__()`
enforcement.)

### Checkpoint 7

1. In the `Doubler` example: which class defines `__call__`, which
   defines `forward`, and how do the two meet when you evaluate `d(x)`?
2. Which exact exception (type and message) does torch raise when a
   subclass assigns an `nn.Parameter` before calling
   `super().__init__()`?
3. Write a module `Negator(nn.Module)` whose `forward` returns `-x`.
   What does `Negator()(torch.tensor([3.0, -1.0]))` print?



## 8. Common Pitfalls I

**Pitfall 1 — trusting the float default across the border.**
Torch code written before the course's `set_default_dtype` line runs in
float32, and float32 values compared against NumPy's float64 fail
tight asserts:


In [ ]:

vals32 = torch.full((1000,), 0.1, dtype=torch.float32)   # deliberately float32
total32 = vals32.sum().item()
expected = 0.1 * 1000
print("float32 total:", f"{total32:.10f}", " expected:", expected)
print("tight check   (|diff| < 1e-12):", abs(total32 - expected) < 1e-12)
print("stated tolerance (atol=1e-6 * n):", abs(total32 - expected) < 1e-6 * 1000)

total64 = torch.full((1000,), 0.1).sum().item()          # float64 under the course default
print("float64 total passes tight check:", abs(total64 - expected) < 1e-9)



The float32 sum lands at `100.0000076294` — the tight check prints
`False`, the stated-tolerance check `True`, and the float64 rerun
`True`.
The fix is the Section 2 convention: float64 by default, and when
float32 is deliberate, say `atol=1e-6`/`rtol=1e-5` out loud.

**Pitfall 2 — integer tensors refuse float jobs.**
`torch.arange` yields `int64`, and integer tensors will not compute a
mean:


In [ ]:

k = torch.arange(5)
try:
    k.mean()
except RuntimeError as e:
    print("RuntimeError:", e)

print("fixed:", k.to(torch.float64).mean())    # cast first -> tensor(2.)



The error message names the problem (`mean` cannot run on `Long`, i.e.
int64 — it needs a floating or complex dtype); the one-word fix is an
explicit cast, `k.to(torch.float64)`, after which the mean prints
`tensor(2.)`.
When you *divide* integer tensors you get floats automatically
(`torch.arange(5) / 2` is floating) — but reductions like `mean` make
you choose the dtype yourself.

**Pitfall 3 — the unrepeatable "reproducible" run.**
Seeding once at the top of a notebook does not make every later cell
reproducible: each random call advances the stream, so re-running a
*middle* cell draws different numbers.


In [ ]:

torch.manual_seed(20260804)
first = torch.randn(3)
second = torch.randn(3)                 # different from first: the stream moved on
print("first :", first)
print("second:", second)

# discipline: re-seed inside any cell whose numbers must be replayable
torch.manual_seed(20260804)
replay = torch.randn(3)
print("replay equals first:", bool((replay == first).all()))



`second` differs from `first`; the re-seeded `replay` matches `first`
exactly (`True`).
Course discipline: `torch.manual_seed(20260804)` appears **in the same
cell as** every draw a narration or an assert depends on — the habit
every seeded notebook in this unit follows.

### Checkpoint 8

1. A teammate's float32 forward pass fails
   `abs(out - expected) < 1e-10` by about `3e-8`.
   Is the code wrong? What are the two course-sanctioned repairs?
2. `torch.arange(6).mean()` raises a `RuntimeError`.
   Why, and what is the fix?
3. A notebook seeds once in cell 1; cell 5 draws `torch.randn(4)`.
   Re-running only cell 5 gives new numbers every time.
   Explain, and state the discipline that prevents it.



## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. Shape `torch.Size([2, 5])`, `ndim = 2`; `u[1, 3]` is the 4th entry
   of the 2nd row: rows are `0..4` and `5..9`, so `u[1, 3] = 8`
   (a 0-d tensor; `.item()` gives the plain `8`).
2. `torch.zeros(4, 2)` (or `torch.zeros((4, 2))`); `b.sum(dim=1)`.
3. `tensor(2.)` — extract with `v[0, 1].item()`.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. `torch.int64` and `torch.float32` — the float32 default is the
   NumPy difference to memorize.
2. Float32 carries ~7 significant digits, so its result differs from
   the float64 reference by far more than `1e-12`.
   Fixes: compute in float64 (`torch.set_default_dtype(torch.float64)`
   or explicit `.to(torch.float64)` casts), or keep float32 and state
   the honest tolerance `atol=1e-6`/`rtol=1e-5`.
3. Still `torch.int64`: `set_default_dtype` governs only floating
   dtypes; integer literals always build int64 tensors.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. `tensor([ 6., 15.])`, shape `(2,)` — `dim=1` collapses the columns,
   one sum per row.
2. `(5, 3)`: broadcasting aligns trailing dimensions, stretching `v`
   across the 5 rows — F1's trailing-dimensions rule, unchanged in
   torch.
3. `int((t > 5).sum())` — the entries `6..11`, so `6`.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. `tensor([1., 5.])` — `from_numpy` shares memory with `arr`, so the
   write through the NumPy name is visible through the torch name.
2. No — the second call continues the stream.
   Re-run `torch.manual_seed(20260804)` immediately before the second
   draw to replay the first.
3. The explicit cast makes the boundary dtype an invariant of the code
   instead of an accident of the input: fed a float32 array, the
   uncast version would silently produce a float32 tensor and break
   downstream float64 arithmetic or asserts.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. ```python
   class Counter:
       def __init__(self):
           self.n = 0
       def bump(self):
           self.n += 1
           return self.n
   ```
   Two calls return `1` then `2` — the object remembers `self.n`
   between calls.
2. `lab` becomes `self`; the list attribute `self.readings` gains the
   value `24.0`.
3. `lab.unit` is an attribute (a stored value, here `"C"`);
   `lab.summary` is a method (a bound function).
   The parentheses *call* it — without them you have the function
   object itself, not its result.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. `apply(3.0)` returns `6.0`; `twice(3.0)` returns `12.0`
   ($3 \to 6 \to 12$ through the inherited `twice`); `label()` returns
   `"scale-transform"`.
2. `Shift.apply`.
   Method lookup starts at the *object's own class* and only falls back
   to the base — so `self.apply` inside inherited code finds the
   override.
   (This is the mechanism that will let `nn.Module`'s machinery call
   *your* `forward`.)
3. `False` and `True`: `Scale` and `Shift` are siblings, but both are
   `Transform`s.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. `nn.Module` defines `__call__`; `Doubler` defines `forward`.
   Evaluating `d(x)` triggers the inherited `__call__`, which runs the
   base class's housekeeping and then dispatches to the subclass's
   `forward` — Section 6's lookup rule doing real work.
2. `AttributeError: cannot assign parameters before Module.__init__()
   call`.
3. ```python
   class Negator(nn.Module):
       def forward(self, x):
           return -x
   ```
   It prints `tensor([-3.,  1.])`.
   (Defining no `__init__` is legal — the base class's `__init__` runs
   automatically; Session 2 leans on this for parameter-free modules.)

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. The code is fine — `3e-8` is normal float32 rounding, invisible only
   to an unrealistic `1e-10` gate.
   Repairs: run the computation in float64, or keep float32 and loosen
   the check to the stated `atol=1e-6`/`rtol=1e-5`.
2. `arange` builds an `int64` tensor and `mean` refuses integer input;
   fix with `torch.arange(6).to(torch.float64).mean()`.
3. Cell 1's seed fixes the start of one global stream; by the time
   cell 5 first ran, earlier draws had advanced it, and each re-run of
   cell 5 advances it further.
   Discipline: put `torch.manual_seed(20260804)` in the same cell as
   the draw that must be replayable.

</details>
